# 5. Evaluation: From Gut Checks to Automated Judgment

In [2]:
import sys 
sys.path.insert(0, "..") 
from config import API_KEY as key, ENDPOINT_BASE as endpoint_base 
import requests 
import json 
from datetime import datetime

## 5.1 The Manual Phase: Ask Questions, Write Down What's Wrong

This is how every customer starts. Someone sits in front of the system, asks a few questions, and forms an opinion. That opinion usually sounds like one of these:

* "It kind of works."
* "Some answers are good, some are weird."
* "I don't trust it yet."

We're going to do the same thing, deliberately. But unlike the customer, we're going to be structured about it. We already have two questions from earlier in the lab. Now we're going to expand that to a small set, run each question two ways, without context and with RAG,  and record what we observe.

This isn't sophisticated. It's not meant to be. The point is to do what the customer does, feel why it breaks down, and understand what we'd need to do it better.


In [3]:
eval_questions = [
    {
        "id": "q1",
        "question": "What happens if a Thief fails an Open Locks attempt?",
        "known_answer": "The Thief may only try once per lock. If they fail, they must wait until they gain another level of experience before trying again.",
        "why": "Control question — we already verified this in Section 4."
    },
    {
        "id": "q2",
        "question": "Why can't Elves roll higher than a d6 for hit points?",
        "known_answer": "Elves use a d6 for hit dice because of their character class restrictions in Basic Fantasy RPG.",
        "why": "Tests whether the model answers from this game or defaults to D&D."
    },
    {
        "id": "q3",
        "question": "Can a Fighter use magic-user scrolls?",
        "known_answer": "No. Only Magic-Users (and in some cases Thieves at higher levels) can use magic-user scrolls.",
        "why": "Requires interpreting class restriction rules — not a single sentence answer."
    },
    {
        "id": "q4",
        "question": "How is initiative determined in combat?",
        "known_answer": "Each side rolls 1d6 at the start of each round. The side with the higher roll acts first.",
        "why": "A rule that differs significantly across RPG systems. Tests domain specificity."
    },
    {
        "id": "q5",
        "question": "What is the maximum number of retainers a character can hire?",
        "known_answer": "This depends on the character's Charisma score. The Charisma table defines the maximum number of retainers.",
        "why": "Answer requires connecting two pieces of information — Charisma rules and retainer rules."
    },
]

print(f"Evaluation set: {len(eval_questions)} questions loaded")
for q in eval_questions:
    print(f"  [{q['id']}] {q['question']}")

Evaluation set: 5 questions loaded
  [q1] What happens if a Thief fails an Open Locks attempt?
  [q2] Why can't Elves roll higher than a d6 for hit points?
  [q3] Can a Fighter use magic-user scrolls?
  [q4] How is initiative determined in combat?
  [q5] What is the maximum number of retainers a character can hire?


In [6]:
url_chat = f"{endpoint_base}/chat/completions"
model_id = "granite-3-2-8b-instruct"

def ask_without_context(question, model=model_id):
    """Baseline: model answers from its own training data only."""
    headers = {
        "Authorization": f"Bearer {key}",
        "Content-Type": "application/json"
    }
    data = {
        "model": model,
        "messages": [{"role": "user", "content": question}],
        "max_tokens": 300,
        "temperature": 0
    }
    try:
        response = requests.post(url_chat, headers=headers, json=data)
        response.raise_for_status()
        return response.json()["choices"][0]["message"]["content"]
    except Exception as e:
        return f"ERROR: {e}"


def ask_with_rag(question, collection, model=model_id, n_results=3):
    """RAG: retrieve relevant chunks, then ask the model."""
    # Retrieve
    results = collection.query(query_texts=[question], n_results=n_results)
    context_chunks = results["documents"][0]
    context = "\n\n---\n\n".join(context_chunks)

    # Build prompt
    system_msg = (
        "Answer the question using only the provided context. "
        "If the context does not contain enough information, say so."
    )
    headers = {
        "Authorization": f"Bearer {key}",
        "Content-Type": "application/json"
    }
    data = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}
        ],
        "max_tokens": 300,
        "temperature": 0
    }
    try:
        response = requests.post(url_chat, headers=headers, json=data)
        response.raise_for_status()
        answer = response.json()["choices"][0]["message"]["content"]
        return answer, context_chunks
    except Exception as e:
        return f"ERROR: {e}", context_chunks


In [8]:
results = []

print("Running evaluation set...\n")
for q in eval_questions:
    print(f"[{q['id']}] {q['question']}")

    # Baseline (no context)
    baseline_answer = ask_without_context(q["question"])

    # RAG (with retrieved context)
    rag_answer, retrieved_chunks = ask_with_rag(q["question"], collection)

    result = {
        "id": q["id"],
        "question": q["question"],
        "known_answer": q["known_answer"],
        "baseline_answer": baseline_answer,
        "rag_answer": rag_answer,
        "retrieved_chunks": retrieved_chunks,
        "timestamp": datetime.now().isoformat()
    }
    results.append(result)
    print(f"  ✅ Done\n")

print(f"All {len(results)} questions complete.")


Running evaluation set...

[q1] What happens if a Thief fails an Open Locks attempt?


NameError: name 'collection' is not defined